# R5.6 — Is the gain from better labels or from the downstream model?

Reviewer 5, comment 6: evaluate the cleaned datasets with several classifiers under one
identical protocol, against the noisy data and an oracle-clean dataset.

Training sets, all from the same published noisy labels:
`noisy` (as observed) · `ours` (DetectAndRelabel at the oracle thresholds `td`, `tr`) ·
`oracle_filtered` (every truly noisy sample removed: separates *fewer samples* from *better
labels*) · `oracle_clean` (all ground-truth labels: the ceiling).

Classifiers: PreAct-ResNet34 (the paper's), ResNet-18, VGG-16-BN, MobileNetV2 (32×32 stems).
Protocol = the paper's `FinalModelTester` settings for the dataset (Adam, 5-epoch linear warm-up
then constant rate, label smoothing 0.1, stratified validation split, early stopping on
validation accuracy, best weights, clean test set), applied unchanged to every classifier and
training set.

The `ours` × `preact_resnet34` cells follow the paper's Table 3 protocol, so with
`preds_ref='main'` they are also the downstream numbers for the re-run CIFAR-10 predictions
(the paper's Table 3 still reports the round-3 run, `preds_ref='513d833'`).

**Budget (rough):** CIFAR-10 ≈ 0.4–0.8 h per job on a T4; 4 sets × 4 classifiers × 3 seeds = 48 jobs
≈ 30 GPU-h ≈ 15 h on T4×2. Jobs run seed by seed, so each commit leaves complete seeds.

**Kaggle settings:** Accelerator *GPU T4 x2* (P100 also works, one worker), Internet *On*
(phone-verified account), Persistence *Files only*. Run with **Save Version -> Save & Run All
(Commit)** so it runs in the background for up to 12 h.

**Resuming:** jobs are saved as they finish. If the session ends with jobs left, open the
notebook, *Add Input -> Your Work -> this notebook* (the previous version's output), and
commit again: finished jobs are restored and skipped. Repeat until the launch cell prints
that nothing is left; the analysis cell then reports the final tables.

**Outputs** (in the version's Output tab): `r56_out/` raw per-job results, `results/` CSV
tables, `logs/` per-GPU training logs.

## 1. Configuration

In [ ]:
import json, os
# Oracle thresholds (td, tr) per run. 'main' = the CIFAR-10 re-run (best noise-F1 on the paper's
# 6..10 grid with tr=10 -- confirm before use); '513d833' = the round-3 paper run (its Table 2).
# Fashion-MNIST predictions are the same in both.
THRESHOLDS = {
    ('main', 'cifar10', 20): (10, 10), ('main', 'cifar10', 30): (8, 10), ('main', 'cifar10', 40): (8, 10),
    ('513d833', 'cifar10', 20): (8, 10), ('513d833', 'cifar10', 30): (7, 10), ('513d833', 'cifar10', 40): (6, 10),
    ('main', 'fashionmnist', 20): (9, 10), ('main', 'fashionmnist', 30): (9, 10), ('main', 'fashionmnist', 40): (7, 10),
}
CONFIG = dict(
    dataset='cifar10', noise=20, preds_ref='main', code_ref='main',
    train_sets=['noisy', 'ours', 'oracle_filtered', 'oracle_clean'],
    archs=['preact_resnet34', 'resnet18', 'vgg16_bn', 'mobilenet_v2'],
    seeds=[0, 1, 2], amp=True,
    session_hours=11.5, job_hours=0.9, loader_workers=2,
)
CONFIG.update(json.loads(os.environ.get('SND_CONFIG_OVERRIDES', '{}')))   # local smoke tests only
CONFIG['td'], CONFIG['tr'] = THRESHOLDS[(CONFIG['preds_ref'], CONFIG['dataset'], CONFIG['noise'])]
OUT_NAME, WORKER, PIP = 'r56_out', 'r56_worker.py', ['timm']
N_JOBS = len(CONFIG['train_sets']) * len(CONFIG['archs']) * len(CONFIG['seeds'])

## 2. Helper modules (written to the working directory)

In [ ]:
%%writefile snd_kaggle.py
"""Shared plumbing for the round-4 revision experiments (Kaggle or any CUDA box).

What lives here:
  * repo setup: clone the public SiameseNoiseDetection repo, import `snd` from it;
  * the exact noisy labels and outer folds of a published run, rebuilt from its
    committed prediction CSVs (the IDN generator re-randomises, so the CSVs are the
    only faithful record of which labels were corrupted);
  * the per-dataset protocol as actually run (main.ipynb cells 16/25/35, 59/71/80,
    106-124), which differs in places from snd/config.py and the CLI;
  * a crash-safe job queue that keeps every visible GPU busy and survives Kaggle's
    12-hour session limit by resuming from a previous version's output.
"""
import glob
import json
import os
import re
import shutil
import subprocess
import sys
import time

import numpy as np
import pandas as pd

REPO_URL = 'https://github.com/stormaref/SiameseNoiseDetection.git'
ON_KAGGLE = os.path.isdir('/kaggle/working')
WORK = '/kaggle/working' if ON_KAGGLE else os.path.abspath(os.environ.get('SND_WORK', 'work'))
# repo clone, extracted preds and datasets are re-creatable: keep them out of the saved output
SCRATCH = '/tmp/snd' if ON_KAGGLE else WORK
DATA_ROOT = os.path.join(SCRATCH, 'data')

CIFAR_MEAN, CIFAR_STD = (0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)

# Siamese-ensemble protocol per published run (main.ipynb; values that differ
# from snd/config.py are noted). `aug` names an entry of `augmentation()` below.
PROTOCOL = {
    ('cifar10', 20): dict(preds='cifar10(20)', backbone='resnet50', pre_trained=True, emb=64,
                          wd=5e-4, patience=8, margin=2, aug='cifar_crop'),
    ('cifar10', 30): dict(preds='cifar10(30)', backbone='resnet50', pre_trained=True, emb=64,
                          wd=5e-4, patience=10, margin=2, aug='cifar_crop'),
    ('cifar10', 40): dict(preds='cifar10(40)', backbone='resnet50', pre_trained=True, emb=64,
                          wd=5e-4, patience=15, margin=2, aug='cifar_affine'),
    ('fashionmnist', 20): dict(preds='fmnist(20)', backbone='resnet34', pre_trained=False, emb=128,
                               wd=1e-3, patience=12, margin=2, aug='fmnist_plain'),
    ('fashionmnist', 30): dict(preds='fmnist(30)', backbone='resnet34', pre_trained=False, emb=128,
                               wd=1e-3, patience=12, margin=2, aug='fmnist_plain'),
    ('fashionmnist', 40): dict(preds='fmnist(40)', backbone='resnet34', pre_trained=False, emb=128,
                               wd=1e-3, patience=12, margin=2, aug='fmnist_norm'),
}
# Shared by every run above.
SIAMESE_COMMON = dict(lr=5e-5, batch_size=2048, train_pairs=200_000, val_pairs=20_000,
                      dropout=0.5, label_smoothing=0.1, max_epochs=1000, inner_folds=10)

# Downstream-classifier protocol (FinalEvaluator calls, main.ipynb cells 106-124).
DOWNSTREAM = {
    'cifar10': dict(lr=1e-3, wd=5.46e-5, batch_size=256, patience=20, warmup=5,
                    smoothing=0.1, val_ratio=0.1, max_epochs=200),
    'fashionmnist': dict(lr=2e-4, wd=1.12e-6, batch_size=256, patience=10, warmup=5,
                         smoothing=0.1, val_ratio=0.05, max_epochs=200),
}


# --------------------------------------------------------------------------- repo
def setup_repo(code_ref='main', repo_dir=None):
    """Clone (or reuse) the code repo, check out `code_ref`, make `snd` importable."""
    repo_dir = repo_dir or os.environ.get('SND_REPO') or os.path.join(SCRATCH, 'SiameseNoiseDetection')
    if not os.path.isdir(os.path.join(repo_dir, '.git')):
        os.makedirs(os.path.dirname(repo_dir), exist_ok=True)
        subprocess.run(['git', 'clone', '--quiet', REPO_URL, repo_dir], check=True)
    if not os.environ.get('SND_REPO'):      # never move the HEAD of a local working copy
        subprocess.run(['git', '-C', repo_dir, 'checkout', '--quiet', code_ref], check=True)
    # local testing: SND_SRC points at an extracted `src/` of code_ref when the working
    # copy in SND_REPO is on another branch
    src = os.environ.get('SND_SRC') or os.path.join(repo_dir, 'src')
    if src not in sys.path:
        sys.path.insert(0, src)
    return repo_dir


def _resolve_ref(repo_dir, ref):
    for candidate in (f'origin/{ref}', ref):
        ok = subprocess.run(['git', '-C', repo_dir, 'rev-parse', '--verify', '--quiet', candidate],
                            capture_output=True).returncode == 0
        if ok:
            return candidate
    raise ValueError(f'git ref {ref!r} not found in {repo_dir}')


def extract_preds(repo_dir, preds_ref, preds_name):
    """Materialise preds/<preds_name> as committed on branch `preds_ref`; return its dir."""
    out = os.path.join(SCRATCH, 'preds', preds_ref, preds_name)
    if not glob.glob(os.path.join(out, '*', 'fold*_analysis.csv')):
        os.makedirs(os.path.dirname(out), exist_ok=True)
        ref = _resolve_ref(repo_dir, preds_ref)
        archive = subprocess.run(['git', '-C', repo_dir, 'archive', ref, f'preds/{preds_name}'],
                                 capture_output=True, check=True).stdout
        tmp = out + '.tmp'
        os.makedirs(tmp, exist_ok=True)
        subprocess.run(['tar', '-x', '-C', tmp], input=archive, check=True)
        shutil.move(os.path.join(tmp, 'preds', preds_name), out)
        shutil.rmtree(tmp)
    return out


# --------------------------------------------------------------------------- labels
def load_label_table(preds_dir):
    """One row per training sample: noisy/true label, noise flag, outer fold, saved preds."""
    frames = []
    for path in glob.glob(os.path.join(preds_dir, '*', 'fold*_analysis.csv')):
        fold = int(re.search(r'fold(\d+)_analysis', path).group(1))
        df = pd.read_csv(path)
        df['outer_fold'] = fold
        frames.append(df)
    table = pd.concat(frames).set_index('index').sort_index()
    table['is_noisy'] = table['is_noisy'].astype(str).str.lower().eq('true')
    if not table.index.is_unique:
        raise ValueError(f'{preds_dir}: a sample appears in more than one outer fold')
    return table


def base_dataset(name, train=True, attempts=5):
    from torchvision.datasets import CIFAR10, FashionMNIST
    cls = {'cifar10': CIFAR10, 'fashionmnist': FashionMNIST}[name]
    for attempt in range(attempts):             # the mirrors fail transiently now and then
        try:
            return cls(root=DATA_ROOT, train=train, download=True)
        except RuntimeError as error:
            if attempt == attempts - 1:
                raise
            print(f'download failed ({error}); retrying in 30 s', flush=True)
            time.sleep(30)


def noisy_train_set(name, table):
    """The torchvision training set with its targets replaced by the published noisy labels."""
    ds = base_dataset(name, train=True)
    if len(table) != len(ds) or (table.index.to_numpy() != np.arange(len(ds))).any():
        raise ValueError('prediction CSVs do not cover the training set exactly once')
    true = np.asarray(ds.targets)
    if (table['real_label'].to_numpy() != true).any():
        raise ValueError('real_label in the CSVs does not match torchvision targets')
    ds.targets = [int(y) for y in table['noisy_label']]
    return ds


def smoke_subsample(table, cfg):
    """Local smoke tests only: a fixed random subset, re-indexed 0..n-1 (no-op on real runs)."""
    if not cfg.get('subsample'):
        return table, None
    keep = np.sort(np.random.default_rng(0).choice(len(table), cfg['subsample'], replace=False))
    return table.iloc[keep].reset_index(drop=True), keep


def augmentation(name):
    """Transforms exactly as in main.ipynb (scoring always uses `eval`)."""
    from torchvision import transforms as T
    cifar_norm = [T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)]
    gray = [T.Grayscale(num_output_channels=3), T.ToTensor()]
    table = {
        'cifar_crop': (T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip(0.5),
                                  T.RandomRotation(15), *cifar_norm]),
                       T.Compose(cifar_norm)),
        'cifar_affine': (T.Compose([T.RandomRotation(15), T.RandomHorizontalFlip(0.5),
                                    T.RandomAffine(0, translate=(0.1, 0.1)),
                                    T.RandomResizedCrop(32, scale=(0.9, 1.0)), *cifar_norm]),
                         T.Compose(cifar_norm)),
        'fmnist_plain': (T.Compose(gray), T.Compose(gray)),
        'fmnist_norm': (T.Compose(gray + [T.Normalize((0.5,), (0.5,))]),
                        T.Compose(gray + [T.Normalize((0.5,), (0.5,))])),
        # downstream classifiers (FinalEvaluator cells)
        'down_cifar': (T.Compose([T.RandomRotation(15), T.RandomHorizontalFlip(0.5),
                                  T.RandomAffine(0, translate=(0.1, 0.1)),
                                  T.RandomResizedCrop(32, scale=(0.9, 1.0)), *cifar_norm]),
                       T.Compose(cifar_norm)),
        'down_fmnist': (T.Compose([T.Grayscale(num_output_channels=3), T.RandomHorizontalFlip(0.5),
                                   T.RandomRotation(10), T.RandomAffine(0, translate=(0.1, 0.1)),
                                   T.ToTensor(), T.Normalize((0.5,), (0.5,))]),
                        T.Compose(gray + [T.Normalize((0.5,), (0.5,))])),
    }
    return table[name]


def cleaned_labels(table, td, tr):
    """Apply the published DetectAndRelabel rule (NoiseCleaner.advanced_clean).

    Returns an int array of new labels with -1 for removed samples: flagged if
    mistakes >= td; relabelled to a class predicted by >= tr members, else removed.
    """
    labels = table['noisy_label'].to_numpy().copy()
    for pos, (mistakes, preds) in enumerate(zip(table['mistakes'], table['preds'])):
        if mistakes < td:
            continue
        votes = np.bincount(np.fromstring(preds, dtype=int, sep='|'), minlength=10)
        winners = np.flatnonzero(votes >= tr)
        labels[pos] = winners[0] if len(winners) else -1
    return labels


# --------------------------------------------------------------------------- jobs
class JobQueue:
    """File-backed queue: a job is done when its result file exists.

    Workers claim a job by atomically creating `<result>.claim`; claims left behind by
    a killed session are ignored once older than `stale_hours`.
    """

    def __init__(self, out_dir, stale_hours=13):
        self.out_dir = out_dir
        self.stale = stale_hours * 3600
        os.makedirs(out_dir, exist_ok=True)

    def done(self, job_id):
        return os.path.exists(os.path.join(self.out_dir, job_id + '.done'))

    def claim(self, job_id):
        path = os.path.join(self.out_dir, job_id + '.claim')
        if os.path.exists(path) and time.time() - os.path.getmtime(path) > self.stale:
            os.remove(path)
        try:
            fd = os.open(path, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
        except FileExistsError:
            return False
        os.write(fd, str(os.getpid()).encode())
        os.close(fd)
        return True

    def finish(self, job_id, summary):
        with open(os.path.join(self.out_dir, job_id + '.done'), 'w') as f:
            json.dump(summary, f, indent=1)
        claim = os.path.join(self.out_dir, job_id + '.claim')
        if os.path.exists(claim):
            os.remove(claim)

    def release(self, job_id):
        claim = os.path.join(self.out_dir, job_id + '.claim')
        if os.path.exists(claim):
            os.remove(claim)

    def summaries(self):
        rows = []
        for path in sorted(glob.glob(os.path.join(self.out_dir, '*.done'))):
            with open(path) as f:
                rows.append(json.load(f))
        return rows


def restore_previous_outputs(out_name):
    """Copy `<out_name>/` from any attached earlier notebook version into WORK.

    On Kaggle: Add Input -> Your Work -> this notebook (a previous version); its
    /kaggle/working appears under /kaggle/input/<slug>/.
    """
    target = os.path.join(WORK, out_name)
    os.makedirs(target, exist_ok=True)
    copied = 0
    for src in glob.glob(f'/kaggle/input/*/{out_name}') + glob.glob(f'/kaggle/input/*/*/{out_name}'):
        for path in glob.glob(os.path.join(src, '**', '*'), recursive=True):
            if os.path.isdir(path) or path.endswith('.claim'):
                continue
            dest = os.path.join(target, os.path.relpath(path, src))
            if not os.path.exists(dest):
                os.makedirs(os.path.dirname(dest), exist_ok=True)
                shutil.copy2(path, dest)
                copied += 1
    print(f'restored {copied} files into {target}')
    return target


def launch_workers(script, config_path, n_gpus=None, log_dir=None, poll=60):
    """Run `python script config_path` once per GPU and stream a progress line per poll."""
    import torch
    n_gpus = n_gpus or max(torch.cuda.device_count(), 1)
    log_dir = log_dir or os.path.join(WORK, 'logs')
    os.makedirs(log_dir, exist_ok=True)
    procs = []
    for gpu in range(n_gpus):
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), SND_WORKER=str(gpu))
        log = open(os.path.join(log_dir, f'worker{gpu}.log'), 'a')
        procs.append((subprocess.Popen([sys.executable, '-u', script, config_path], env=env,
                                       stdout=log, stderr=subprocess.STDOUT), log))
        time.sleep(20)                       # stagger dataset downloads / pair generation
    while any(p.poll() is None for p, _ in procs):
        time.sleep(poll)
        for gpu in range(n_gpus):
            tail = _last_line(os.path.join(log_dir, f'worker{gpu}.log'))
            print(f'[{time.strftime("%H:%M")}] gpu{gpu}: {tail}', flush=True)
    for p, log in procs:
        log.close()
    return [p.returncode for p, _ in procs]


def _last_line(path):
    try:
        with open(path, 'rb') as f:
            f.seek(0, 2)
            f.seek(max(f.tell() - 4000, 0))
            lines = f.read().decode(errors='replace').replace('\r', '\n').strip().splitlines()
            return lines[-1][-160:] if lines else ''
    except OSError:
        return ''


class Deadline:
    """Stop taking new jobs when the session is close to Kaggle's 12-hour limit."""

    def __init__(self, hours):
        self.end = time.time() + hours * 3600

    def allows(self, expected_hours):
        return time.time() + expected_hours * 3600 < self.end

In [ ]:
%%writefile r56_worker.py
"""R5.6 worker: train downstream classifiers on noisy / cleaned / oracle training sets.

Training sets (all built from the same published noisy labels):
  noisy            every sample, observed labels;
  ours             DetectAndRelabel output at (td, tr): flagged samples relabelled or removed;
  oracle_filtered  every truly noisy sample removed, the rest keep their (correct) labels;
  oracle_clean     every sample, ground-truth labels.
`oracle_filtered` separates "fewer samples" from "better labels"; `oracle_clean` is the ceiling.

Protocol = snd.evaluation.final_model_tester.FinalModelTester as called in main.ipynb:
stratified train/val split of the training set, Adam, linear warm-up for `warmup` epochs and
then a constant rate, label smoothing, early stopping on validation accuracy, best weights
reloaded, accuracy on the clean test set. Usage: python r56_worker.py <config.json>
"""
import copy
import json
import os
import sys
import time

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import snd_kaggle as K  # noqa: E402

TRAIN_SETS = ('noisy', 'ours', 'oracle_filtered', 'oracle_clean')


def build_arch(name):
    """32x32-style variants of four architecture families (also valid at 28x28)."""
    from torchvision import models
    if name == 'preact_resnet34':                    # the paper's downstream model
        from snd.models.preact import PreActResNet34
        return PreActResNet34()
    if name == 'resnet18':
        m = models.resnet18(num_classes=10)
        m.conv1 = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
        m.maxpool = nn.Identity()
        return m
    if name == 'vgg16_bn':
        m = models.vgg16_bn(num_classes=10)
        m.features[-1] = nn.Identity()               # keep >= 1x1 maps for 28x28 inputs
        m.avgpool = nn.AdaptiveAvgPool2d(1)
        m.classifier = nn.Linear(512, 10)
        return m
    if name == 'mobilenet_v2':
        m = models.mobilenet_v2(num_classes=10)
        m.features[0][0].stride = (1, 1)
        return m
    if name == 'densenet121':
        m = models.densenet121(num_classes=10)
        m.features.conv0 = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
        m.features.pool0 = nn.Identity()
        return m
    raise ValueError(name)


def training_labels(table, kind, td, tr):
    """Label per sample for one training set; -1 = not in the set."""
    if kind == 'noisy':
        return table['noisy_label'].to_numpy().copy()
    if kind == 'ours':
        return K.cleaned_labels(table, td, tr)
    if kind == 'oracle_filtered':
        return np.where(table['is_noisy'].to_numpy(), -1, table['noisy_label'].to_numpy())
    if kind == 'oracle_clean':
        return table['real_label'].to_numpy().copy()
    raise ValueError(kind)


class Relabelled(Dataset):
    def __init__(self, base, indices, labels, transform):
        self.base, self.indices, self.labels, self.transform = base, indices, labels, transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        img, _ = self.base[self.indices[i]]
        return self.transform(img), int(self.labels[i])


def evaluate(model, loader, device, amp):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device)
            with torch.autocast('cuda', dtype=torch.float16, enabled=amp):
                out = model(x)
            correct += (out.argmax(1) == y).sum().item()
            total += len(y)
    return correct / total


def run_job(cfg, job, base_train, base_test, table):
    from sklearn.model_selection import train_test_split
    p = dict(K.DOWNSTREAM[cfg['dataset']], **cfg.get('overrides', {}))
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    amp = cfg['amp'] and device.type == 'cuda'
    torch.manual_seed(job['seed'])
    np.random.seed(job['seed'])

    labels = training_labels(table, job['train_set'], cfg['td'], cfg['tr'])
    keep = np.flatnonzero(labels >= 0)
    true = table['real_label'].to_numpy()
    tr_pos, va_pos = train_test_split(np.arange(len(keep)), test_size=p['val_ratio'],
                                      stratify=labels[keep], random_state=job['seed'])
    aug, plain = K.augmentation('down_cifar' if cfg['dataset'] == 'cifar10' else 'down_fmnist')
    workers = cfg.get('loader_workers', 2)
    mk = lambda pos, t, shuffle, bs: DataLoader(  # noqa: E731
        Relabelled(base_train, keep[pos], labels[keep][pos], t), batch_size=bs, shuffle=shuffle,
        num_workers=workers, pin_memory=True, persistent_workers=workers > 0)
    train_loader = mk(tr_pos, aug, True, p['batch_size'])
    val_loader = mk(va_pos, plain, False, 512)
    test_loader = DataLoader(Relabelled(base_test, np.arange(len(base_test)),
                                        np.asarray(base_test.targets), plain),
                             batch_size=512, num_workers=workers)

    model = build_arch(job['arch']).to(device).to(memory_format=torch.channels_last)
    opt = torch.optim.Adam(model.parameters(), lr=p['lr'], betas=(0.9, 0.999), eps=1e-8,
                           weight_decay=p['wd'])
    warmup = torch.optim.lr_scheduler.LinearLR(opt, start_factor=0.1, total_iters=p['warmup'])
    scaler = torch.amp.GradScaler('cuda', enabled=amp)
    ce = nn.CrossEntropyLoss(label_smoothing=p['smoothing'])

    best, best_state, stale, history = -1.0, None, 0, []
    t0 = time.time()
    for epoch in range(p['max_epochs']):
        model.train()
        loss_sum = 0.0
        for x, y in train_loader:
            x = x.to(device, non_blocking=True).to(memory_format=torch.channels_last)
            y = y.to(device, non_blocking=True)
            with torch.autocast('cuda', dtype=torch.float16, enabled=amp):
                loss = ce(model(x).float(), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            loss_sum += loss.item()
        val_acc = evaluate(model, val_loader, device, amp)
        history.append([epoch, loss_sum / len(train_loader), val_acc])
        if val_acc > best:
            best, stale, best_state = val_acc, 0, copy.deepcopy(model.state_dict())
        else:
            stale += 1
        print(f'{job["id"]} ep{epoch} loss {loss_sum / len(train_loader):.3f} val {val_acc:.4f} '
              f'best {best:.4f} stale {stale} [{(time.time() - t0) / 60:.0f} min]', flush=True)
        if stale >= p['patience']:
            break
        if epoch < p['warmup']:
            warmup.step()
    model.load_state_dict(best_state)
    test_acc = evaluate(model, test_loader, device, amp)
    kept_labels = labels[keep]
    return dict(job=job['id'], dataset=cfg['dataset'], noise=cfg['noise'], preds_ref=cfg['preds_ref'],
                td=cfg['td'], tr=cfg['tr'], train_set=job['train_set'], arch=job['arch'], seed=job['seed'],
                n_train_total=int(len(keep)), n_removed=int(len(labels) - len(keep)),
                label_noise_in_set=float((kept_labels != true[keep]).mean()),
                best_val_acc=best, test_acc=test_acc, epochs=len(history),
                minutes=round((time.time() - t0) / 60, 1), history=history)


def jobs_for(cfg):
    return [dict(id=f'{cfg["dataset"]}{cfg["noise"]}_{s}_{a}_s{seed}', train_set=s, arch=a, seed=seed)
            for seed in cfg['seeds'] for a in cfg['archs'] for s in cfg['train_sets']]


def main(config_path):
    with open(config_path) as f:
        cfg = json.load(f)
    repo = K.setup_repo(cfg['code_ref'])
    proto = K.PROTOCOL[(cfg['dataset'], cfg['noise'])]
    table = K.load_label_table(K.extract_preds(repo, cfg['preds_ref'], proto['preds']))
    base_train = K.noisy_train_set(cfg['dataset'], table)     # images only; labels come from the table
    base_test = K.base_dataset(cfg['dataset'], train=False)
    table, keep = K.smoke_subsample(table, cfg)
    if keep is not None:
        base_train = torch.utils.data.Subset(base_train, keep)
        base_test = torch.utils.data.Subset(base_test, np.arange(cfg['subsample'] // 5))
        base_test.targets = np.asarray(base_test.dataset.targets)[base_test.indices]

    queue = K.JobQueue(cfg['out_dir'])
    deadline = K.Deadline(cfg['session_hours'] - (time.time() - cfg['session_start']) / 3600)
    for job in jobs_for(cfg):
        if queue.done(job['id']):
            continue
        if not deadline.allows(cfg['job_hours']):
            print('session deadline reached; remaining jobs left for the next version', flush=True)
            break
        if not queue.claim(job['id']):
            continue
        try:
            summary = run_job(cfg, job, base_train, base_test, table)
        except BaseException:
            queue.release(job['id'])
            raise
        queue.finish(job['id'], summary)
        print('DONE', json.dumps({k: v for k, v in summary.items() if k != 'history'}), flush=True)


if __name__ == '__main__':
    main(sys.argv[1])

In [ ]:
%%writefile r56_analysis.py
"""R5.6 analysis: does the cleaning gain hold across downstream classifiers?

Reads every <job>.done written by r56_worker.py and reports, per (dataset, noise, arch):
test accuracy (mean +- std over seeds) for each training set, the gain of `ours` over
`noisy`, and the share of the achievable gain it recovers, (ours - noisy) / (oracle_clean - noisy).
`oracle_filtered` vs `oracle_clean` shows what removal alone costs.
"""
import glob
import json
import os

import pandas as pd

ORDER = ['noisy', 'ours', 'oracle_filtered', 'oracle_clean']


def load(out_dir):
    rows = []
    for path in glob.glob(os.path.join(out_dir, '*.done')):
        with open(path) as f:
            rows.append({k: v for k, v in json.load(f).items() if k != 'history'})
    return pd.DataFrame(rows)


def summarise(df):
    keys = ['dataset', 'noise', 'arch']
    acc = (df.groupby(keys + ['train_set'])['test_acc']
             .agg(['mean', 'std', 'count']).reset_index())
    acc['cell'] = acc.apply(lambda r: f"{100 * r['mean']:.2f} ± {100 * (r['std'] if r['count'] > 1 else 0):.2f}"
                            f" (n={int(r['count'])})", axis=1)
    table = acc.pivot_table(index=keys, columns='train_set', values='cell', aggfunc='first')
    table = table[[c for c in ORDER if c in table.columns]]

    means = acc.pivot_table(index=keys, columns='train_set', values='mean')
    gains = pd.DataFrame(index=means.index)
    if {'noisy', 'ours'} <= set(means.columns):
        gains['ours - noisy (pp)'] = 100 * (means['ours'] - means['noisy'])
    if {'noisy', 'ours', 'oracle_clean'} <= set(means.columns):
        gains['share of oracle gain'] = (means['ours'] - means['noisy']) / (means['oracle_clean'] - means['noisy'])
    if {'oracle_filtered', 'oracle_clean'} <= set(means.columns):
        gains['removal cost (pp)'] = 100 * (means['oracle_clean'] - means['oracle_filtered'])
    sets = (df.groupby(['dataset', 'noise', 'train_set'])[['n_train_total', 'label_noise_in_set']]
              .first().reset_index())
    return table, gains, sets


def report(out_dir, results_dir):
    os.makedirs(results_dir, exist_ok=True)
    df = load(out_dir)
    if df.empty:
        print('no finished jobs yet')
        return None
    df.to_csv(os.path.join(results_dir, 'r56_runs.csv'), index=False)
    table, gains, sets = summarise(df)
    table.to_csv(os.path.join(results_dir, 'r56_accuracy.csv'))
    gains.to_csv(os.path.join(results_dir, 'r56_gains.csv'))
    pd.set_option('display.width', 220)
    print('Training sets (size, share of wrong labels):')
    print(sets.round(4).to_string(index=False))
    print('\nTest accuracy (%), mean ± std over seeds:')
    print(table.to_string())
    print('\nGains:')
    print(gains.round(3).to_string())
    return df, table, gains

## 3. Setup

In [ ]:
import json, os, subprocess, sys, time
SESSION_START = time.time()
os.chdir('/kaggle/working' if os.path.isdir('/kaggle/working') else os.getcwd())
subprocess.run('nvidia-smi -L', shell=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + PIP, check=False)
sys.path.insert(0, os.getcwd())
import snd_kaggle as K
repo = K.setup_repo(CONFIG['code_ref'])
print('code:', subprocess.run(['git', '-C', repo, 'log', '-1', '--format=%h %ad %s', '--date=short'],
                              capture_output=True, text=True).stdout)

## 4. Train (all GPUs, resumable)

In [ ]:
for train in (True, False):                     # download once, before the workers start
    K.base_dataset(CONFIG['dataset'], train)
CONFIG['out_dir'] = K.restore_previous_outputs(OUT_NAME)
CONFIG['session_start'] = SESSION_START
with open('config.json', 'w') as f:
    json.dump(CONFIG, f, indent=1)
codes = K.launch_workers(WORKER, 'config.json', poll=CONFIG.get('poll', 300))
print('worker exit codes:', codes)
done = len(K.JobQueue(CONFIG['out_dir']).summaries())
print(f'{done} of {N_JOBS} jobs finished' + ('' if done >= N_JOBS else
      ' -- commit a new version with this one attached as input to continue'))

## 5. Analysis

In [ ]:
import r56_analysis as A
_ = A.report(CONFIG['out_dir'], 'results')